##### Copyright 2019 The TensorFlow Authors.

In [ ]:
#@title Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Prévision de séries temporelles

<table class="tfo-notebook-buttons" align="left">
  <td>
    <a target="_blank" href="https://www.tensorflow.org/tutorials/structured_data/time_series"><img src="https://www.tensorflow.org/images/tf_logo_32px.png" />Voir sur TensorFlow.org</a>
  </td>
  <td>
    <a target="_blank" href="https://colab.research.google.com/github/tensorflow/docs/blob/master/site/en/tutorials/structured_data/time_series.ipynb"><img src="https://www.tensorflow.org/images/colab_logo_32px.png" />Exécuter dans Google Colab</a>
  </td>
  <td>
    <a target="_blank" href="https://github.com/tensorflow/docs/blob/master/site/en/tutorials/structured_data/time_series.ipynb"><img src="https://www.tensorflow.org/images/GitHub-Mark-32px.png" />Voir la source sur GitHub</a>
  </td>
  <td>
    <a href="https://storage.googleapis.com/tensorflow_docs/docs/site/en/tutorials/structured_data/time_series.ipynb"><img src="https://www.tensorflow.org/images/download_logo_32px.png" />Télécharger le notebook</a>
  </td>
</table>

Ce tutoriel est une introduction à la prévision de séries temporelles avec TensorFlow. Il construit plusieurs types de modèles, dont des réseaux de neurones convolutifs et récurrents (CNN et RNN).

Il est composé de deux grandes parties, avec des sous-parties :

* Prévision d'un seul pas de temps :
  * Une seule caractéristique.
  * Toutes les caractéristiques.
* Prévision de plusieurs pas de temps :
  * En une fois (*single-shot*) : toutes les prédictions sont faites d'un coup.
  * Autorégressive : une prédiction à la fois, la sortie étant renvoyée en entrée du modèle.

## Mise en place

In [ ]:
import os
import datetime

import IPython
import IPython.display
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import tensorflow as tf

mpl.rcParams['figure.figsize'] = (8, 6)
mpl.rcParams['axes.grid'] = False

## Le jeu de données météo

Ce tutoriel utilise un <a href="https://www.bgc-jena.mpg.de/wetter/" class="external">jeu de données de séries temporelles météorologiques</a> enregistré par l'<a href="https://www.bgc-jena.mpg.de" class="external">Institut Max Planck de biogéochimie</a>.

Ce jeu de données contient 14 caractéristiques différentes, comme la température de l'air, la pression atmosphérique et l'humidité. Elles ont été relevées toutes les 10 minutes, à partir de 2003. Par souci d'efficacité, vous n'utiliserez que les données relevées entre 2009 et 2016. Cette partie du jeu de données a été préparée par François Chollet pour son livre <a href="https://www.manning.com/books/deep-learning-with-python" class="external">Deep Learning with Python</a>.

In [ ]:
zip_path = tf.keras.utils.get_file(
    origin='https://storage.googleapis.com/tensorflow/tf-keras-datasets/jena_climate_2009_2016.csv.zip',
    fname='jena_climate_2009_2016.csv.zip',
    extract=True)
csv_path, _ = os.path.splitext(zip_path)

Ce tutoriel ne traite que de **prédictions horaires** : commencez donc par sous-échantillonner les données, d'intervalles de 10 minutes à des intervalles d'une heure :

In [ ]:
df = pd.read_csv(csv_path)
# Tranche [début:fin:pas] : à partir de l'indice 5, on garde un relevé sur 6.
df = df[5::6]

date_time = pd.to_datetime(df.pop('Date Time'), format='%d.%m.%Y %H:%M:%S')

Jetons un œil aux données. Voici les premières lignes :

In [ ]:
df.head()

Voici l'évolution de quelques caractéristiques au cours du temps :

In [ ]:
plot_cols = ['T (degC)', 'p (mbar)', 'rho (g/m**3)']
plot_features = df[plot_cols]
plot_features.index = date_time
_ = plot_features.plot(subplots=True)

plot_features = df[plot_cols][:480]
plot_features.index = date_time[:480]
_ = plot_features.plot(subplots=True)

### Inspection et nettoyage

Regardez ensuite les statistiques du jeu de données :

In [ ]:
df.describe().transpose()

#### Vitesse du vent

Une chose devrait sauter aux yeux : la valeur `min` des colonnes de vitesse du vent (`wv (m/s)`) et de vitesse maximale (`max. wv (m/s)`). Ce `-9999` est probablement erroné.

Il y a une colonne séparée pour la direction du vent, donc la vitesse devrait être positive (`>=0`). Remplacez-la par des zéros :

In [ ]:
wv = df['wv (m/s)']
bad_wv = wv == -9999.0
wv[bad_wv] = 0.0

max_wv = df['max. wv (m/s)']
bad_max_wv = max_wv == -9999.0
max_wv[bad_max_wv] = 0.0

# Les modifications en place ci-dessus sont répercutées dans le DataFrame.
df['wv (m/s)'].min()

### Ingénierie des caractéristiques

Avant de se lancer dans la construction d'un modèle, il est important de comprendre ses données et de s'assurer qu'on donne au modèle des données au bon format.

#### Vent
La dernière colonne des données, `wd (deg)`, donne la direction du vent en degrés. Les angles ne sont pas de bonnes entrées pour un modèle : 360° et 0° devraient être proches l'un de l'autre et se rejoindre en douceur. La direction ne devrait pas compter si le vent ne souffle pas.

Pour l'instant, la distribution des données de vent ressemble à ceci :

In [ ]:
plt.hist2d(df['wd (deg)'], df['wv (m/s)'], bins=(50, 50), vmax=400)
plt.colorbar()
plt.xlabel('Direction du vent [deg]')
plt.ylabel('Vitesse du vent [m/s]')

Mais le modèle les interprétera plus facilement si vous convertissez les colonnes de direction et de vitesse du vent en un **vecteur** vent :

In [ ]:
wv = df.pop('wv (m/s)')
max_wv = df.pop('max. wv (m/s)')

# Conversion en radians.
wd_rad = df.pop('wd (deg)')*np.pi / 180

# Calcul des composantes x et y du vent.
df['Wx'] = wv*np.cos(wd_rad)
df['Wy'] = wv*np.sin(wd_rad)

# Calcul des composantes x et y du vent maximal.
df['max Wx'] = max_wv*np.cos(wd_rad)
df['max Wy'] = max_wv*np.sin(wd_rad)

La distribution des vecteurs vent est bien plus simple à interpréter correctement pour le modèle :

In [ ]:
plt.hist2d(df['Wx'], df['Wy'], bins=(50, 50), vmax=400)
plt.colorbar()
plt.xlabel('Vent X [m/s]')
plt.ylabel('Vent Y [m/s]')
ax = plt.gca()
ax.axis('tight')

#### Temps

De même, la colonne `Date Time` est très utile, mais pas sous cette forme de chaîne de caractères. Commencez par la convertir en secondes :

In [ ]:
timestamp_s = date_time.map(pd.Timestamp.timestamp)

Comme pour la direction du vent, le temps en secondes n'est pas une entrée utile pour un modèle. S'agissant de données météo, il y a une périodicité quotidienne et annuelle évidente. Il existe de nombreuses façons de traiter la périodicité.

Vous pouvez obtenir des signaux exploitables en utilisant des transformations sinus et cosinus pour obtenir des signaux clairs d'« heure de la journée » et de « moment de l'année » :

In [ ]:
day = 24*60*60
year = (365.2425)*day

df['Day sin'] = np.sin(timestamp_s * (2 * np.pi / day))
df['Day cos'] = np.cos(timestamp_s * (2 * np.pi / day))
df['Year sin'] = np.sin(timestamp_s * (2 * np.pi / year))
df['Year cos'] = np.cos(timestamp_s * (2 * np.pi / year))

In [ ]:
plt.plot(np.array(df['Day sin'])[:25])
plt.plot(np.array(df['Day cos'])[:25])
plt.xlabel('Temps [h]')
plt.title("Signal d'heure de la journée")

Cela donne au modèle accès aux caractéristiques fréquentielles les plus importantes. Ici, vous saviez à l'avance quelles fréquences étaient importantes.

Si vous n'avez pas cette information, vous pouvez déterminer les fréquences importantes en extrayant des caractéristiques par <a href="https://fr.wikipedia.org/wiki/Transformation_de_Fourier_rapide" class="external">transformée de Fourier rapide</a>. Pour vérifier les hypothèses, voici la `tf.signal.rfft` de la température au cours du temps. Notez les pics évidents aux fréquences proches de `1/an` et `1/jour` :

In [ ]:
fft = tf.signal.rfft(df['T (degC)'])
f_per_dataset = np.arange(0, len(fft))

n_samples_h = len(df['T (degC)'])
hours_per_year = 24*365.2524
years_per_dataset = n_samples_h/(hours_per_year)

f_per_year = f_per_dataset/years_per_dataset
plt.step(f_per_year, np.abs(fft))
plt.xscale('log')
plt.ylim(0, 400000)
plt.xlim([0.1, max(plt.xlim())])
plt.xticks([1, 365.2524], labels=['1/an', '1/jour'])
_ = plt.xlabel('Fréquence (échelle log)')

### Séparation des données

Vous utiliserez une séparation `(70 %, 20 %, 10 %)` pour les ensembles d'entraînement, de validation et de test. Notez que les données ne sont **pas** mélangées aléatoirement avant la séparation, pour deux raisons :

1. Cela garantit qu'il reste possible de découper les données en fenêtres d'exemples consécutifs.
2. Cela garantit que les résultats de validation et de test sont plus réalistes, puisqu'ils sont évalués sur des données relevées après celles sur lesquelles le modèle a été entraîné.

In [ ]:
column_indices = {name: i for i, name in enumerate(df.columns)}

n = len(df)
train_df = df[0:int(n*0.7)]
val_df = df[int(n*0.7):int(n*0.9)]
test_df = df[int(n*0.9):]

num_features = df.shape[1]

### Normalisation des données

Il est important de mettre les caractéristiques à l'échelle avant d'entraîner un réseau de neurones. La normalisation est une façon courante de le faire : soustraire la moyenne et diviser par l'écart-type de chaque caractéristique.

La moyenne et l'écart-type ne doivent être calculés que sur les données d'entraînement, pour que les modèles n'aient pas accès aux valeurs des ensembles de validation et de test.

On pourrait aussi soutenir que le modèle ne devrait pas avoir accès aux valeurs futures de l'ensemble d'entraînement pendant l'entraînement, et que cette normalisation devrait se faire avec des moyennes glissantes. Ce n'est pas le sujet de ce tutoriel, et les ensembles de validation et de test garantissent des métriques (à peu près) honnêtes. Par souci de simplicité, ce tutoriel utilise donc une moyenne simple.

In [ ]:
train_mean = train_df.mean()
train_std = train_df.std()

train_df = (train_df - train_mean) / train_std
val_df = (val_df - train_mean) / train_std
test_df = (test_df - train_mean) / train_std

Jetez maintenant un œil à la distribution des caractéristiques. Certaines ont de longues queues, mais il n'y a pas d'erreur évidente comme la valeur `-9999` de la vitesse du vent.

In [ ]:
df_std = (df - train_mean) / train_std
df_std = df_std.melt(var_name='Column', value_name='Normalized')
plt.figure(figsize=(12, 6))
ax = sns.violinplot(x='Column', y='Normalized', data=df_std)
_ = ax.set_xticklabels(df.keys(), rotation=90)

## Fenêtrage des données

Les modèles de ce tutoriel font un ensemble de prédictions à partir d'une fenêtre d'exemples consécutifs des données.

Les principales caractéristiques des fenêtres d'entrée sont :

- La largeur (nombre de pas de temps) des fenêtres d'entrée et d'étiquettes.
- Le décalage temporel entre elles.
- Les caractéristiques utilisées comme entrées, comme étiquettes, ou les deux.

Ce tutoriel construit une variété de modèles (dont des modèles linéaires, DNN, CNN et RNN), et les utilise pour :

- Des prédictions à *une sortie* et à *plusieurs sorties*.
- Des prédictions d'*un seul pas de temps* et de *plusieurs pas de temps*.

Cette section se concentre sur l'implémentation du fenêtrage des données, pour pouvoir le réutiliser pour tous ces modèles.

Selon la tâche et le type de modèle, vous pourrez vouloir générer différentes fenêtres de données. Voici quelques exemples :

1. Par exemple, pour faire une seule prédiction 24 heures dans le futur à partir de 24 heures d'historique, vous pourriez définir une fenêtre comme celle-ci :

  ![Une prédiction 24 heures dans le futur.](https://github.com/tensorflow/docs/blob/master/site/en/tutorials/structured_data/images/raw_window_24h.png?raw=1)

2. Un modèle qui fait une prédiction une heure dans le futur à partir de six heures d'historique aurait besoin d'une fenêtre comme celle-ci :

  ![Une prédiction une heure dans le futur.](https://github.com/tensorflow/docs/blob/master/site/en/tutorials/structured_data/images/raw_window_1h.png?raw=1)

La suite de cette section définit une classe `WindowGenerator`. Cette classe peut :

1. Gérer les indices et décalages comme sur les schémas ci-dessus.
1. Séparer les fenêtres de caractéristiques en paires `(features, labels)`.
2. Afficher le contenu des fenêtres obtenues.
3. Générer efficacement des batchs de ces fenêtres à partir des données d'entraînement, d'évaluation et de test, à l'aide de `tf.data.Dataset`.

### 1. Indices et décalages

Commencez par créer la classe `WindowGenerator`. La méthode `__init__` contient toute la logique nécessaire pour les indices d'entrée et d'étiquettes.

Elle prend aussi en entrée les DataFrames d'entraînement, d'évaluation et de test. Ils seront convertis plus tard en `tf.data.Dataset` de fenêtres.

In [ ]:
class WindowGenerator():
  def __init__(self, input_width, label_width, shift,
               train_df=train_df, val_df=val_df, test_df=test_df,
               label_columns=None):
    # Stockage des données brutes.
    self.train_df = train_df
    self.val_df = val_df
    self.test_df = test_df

    # Calcul des indices des colonnes d'étiquettes.
    self.label_columns = label_columns
    if label_columns is not None:
      self.label_columns_indices = {name: i for i, name in
                                    enumerate(label_columns)}
    self.column_indices = {name: i for i, name in
                           enumerate(train_df.columns)}

    # Calcul des paramètres de la fenêtre.
    self.input_width = input_width
    self.label_width = label_width
    self.shift = shift

    self.total_window_size = input_width + shift

    self.input_slice = slice(0, input_width)
    self.input_indices = np.arange(self.total_window_size)[self.input_slice]

    self.label_start = self.total_window_size - self.label_width
    self.labels_slice = slice(self.label_start, None)
    self.label_indices = np.arange(self.total_window_size)[self.labels_slice]

  def __repr__(self):
    return '\n'.join([
        f'Taille totale de la fenêtre : {self.total_window_size}',
        f"Indices d'entrée : {self.input_indices}",
        f"Indices d'étiquettes : {self.label_indices}",
        f"Nom(s) des colonnes d'étiquettes : {self.label_columns}"])

Voici le code qui crée les 2 fenêtres des schémas du début de cette section :

In [ ]:
w1 = WindowGenerator(input_width=24, label_width=1, shift=24,
                     label_columns=['T (degC)'])
w1

In [ ]:
w2 = WindowGenerator(input_width=6, label_width=1, shift=1,
                     label_columns=['T (degC)'])
w2

### 2. Séparation

À partir d'une liste d'entrées consécutives, la méthode `split_window` les convertit en une fenêtre d'entrées et une fenêtre d'étiquettes.

L'exemple `w2` défini plus haut sera séparé comme ceci :

![La fenêtre initiale contient tous les exemples consécutifs, que l'on sépare en paires (entrées, étiquettes)](https://github.com/tensorflow/docs/blob/master/site/en/tutorials/structured_data/images/split_window.png?raw=1)

Ce schéma ne montre pas l'axe `features` des données, mais cette fonction `split_window` gère aussi les `label_columns`, ce qui permet de l'utiliser pour les exemples à une sortie comme à plusieurs sorties.

In [ ]:
def split_window(self, features):
  inputs = features[:, self.input_slice, :]
  labels = features[:, self.labels_slice, :]
  if self.label_columns is not None:
    labels = tf.stack(
        [labels[:, :, self.column_indices[name]] for name in self.label_columns],
        axis=-1)

  # Le découpage ne préserve pas l'information de forme statique : on fixe donc
  # les formes à la main. Les `tf.data.Datasets` sont ainsi plus faciles à inspecter.
  inputs.set_shape([None, self.input_width, None])
  labels.set_shape([None, self.label_width, None])

  return inputs, labels

WindowGenerator.split_window = split_window

Essayez-la :

In [ ]:
# Empilement de trois tranches, de la longueur de la fenêtre totale.
example_window = tf.stack([np.array(train_df[:w2.total_window_size]),
                           np.array(train_df[100:100+w2.total_window_size]),
                           np.array(train_df[200:200+w2.total_window_size])])

example_inputs, example_labels = w2.split_window(example_window)

print('Toutes les formes sont : (batch, time, features)')
print(f'Forme de la fenêtre : {example_window.shape}')
print(f'Forme des entrées : {example_inputs.shape}')
print(f'Forme des étiquettes : {example_labels.shape}')

En général, les données dans TensorFlow sont rangées dans des tableaux dont l'indice le plus extérieur parcourt les exemples (la dimension « batch »). Les indices du milieu sont la ou les dimensions de « temps » ou d'« espace » (largeur, hauteur). Les indices les plus intérieurs sont les caractéristiques.

Le code ci-dessus a pris un batch de trois fenêtres de 7 pas de temps, avec 19 caractéristiques à chaque pas. Il les sépare en un batch d'entrées de 6 pas de temps et 19 caractéristiques, et une étiquette de 1 pas de temps et 1 caractéristique. L'étiquette n'a qu'une caractéristique car le `WindowGenerator` a été initialisé avec `label_columns=['T (degC)']`. Dans un premier temps, ce tutoriel construit des modèles qui prédisent une seule étiquette de sortie.

### 3. Affichage

Voici une méthode d'affichage qui permet de visualiser simplement la fenêtre séparée :

In [ ]:
w2.example = example_inputs, example_labels

In [ ]:
def plot(self, model=None, plot_col='T (degC)', max_subplots=3):
  inputs, labels = self.example
  plt.figure(figsize=(12, 8))
  plot_col_index = self.column_indices[plot_col]
  max_n = min(max_subplots, len(inputs))
  for n in range(max_n):
    plt.subplot(max_n, 1, n+1)
    plt.ylabel(f'{plot_col} [normed]')
    plt.plot(self.input_indices, inputs[n, :, plot_col_index],
             label='Inputs', marker='.', zorder=-10)

    if self.label_columns:
      label_col_index = self.label_columns_indices.get(plot_col, None)
    else:
      label_col_index = plot_col_index

    if label_col_index is None:
      continue

    plt.scatter(self.label_indices, labels[n, :, label_col_index],
                edgecolors='k', label='Labels', c='#2ca02c', s=64)
    if model is not None:
      predictions = model(inputs)
      plt.scatter(self.label_indices, predictions[n, :, label_col_index],
                  marker='X', edgecolors='k', label='Predictions',
                  c='#ff7f0e', s=64)

    if n == 0:
      plt.legend()

  plt.xlabel('Temps [h]')

WindowGenerator.plot = plot

Ce graphique aligne les entrées, les étiquettes et (plus tard) les prédictions selon l'instant auquel chaque élément correspond :

In [ ]:
w2.plot()

Vous pouvez afficher les autres colonnes, mais la configuration de la fenêtre d'exemple `w2` n'a d'étiquettes que pour la colonne `T (degC)`.

In [ ]:
w2.plot(plot_col='p (mbar)')

### 4. Création de `tf.data.Dataset`

Enfin, cette méthode `make_dataset` prend un DataFrame de série temporelle et le convertit en `tf.data.Dataset` de paires `(input_window, label_window)` à l'aide de la fonction `tf.keras.utils.timeseries_dataset_from_array` :

In [ ]:
def make_dataset(self, data):
  data = np.array(data, dtype=np.float32)
  ds = tf.keras.utils.timeseries_dataset_from_array(
      data=data,
      targets=None,
      sequence_length=self.total_window_size,
      sequence_stride=1,
      shuffle=True,
      batch_size=32,)

  ds = ds.map(self.split_window)

  return ds

WindowGenerator.make_dataset = make_dataset

L'objet `WindowGenerator` contient les données d'entraînement, de validation et de test.

Ajoutez des propriétés pour y accéder sous forme de `tf.data.Dataset` grâce à la méthode `make_dataset` définie plus haut. Ajoutez aussi un batch d'exemple standard pour y accéder et l'afficher facilement :

In [ ]:
@property
def train(self):
  return self.make_dataset(self.train_df)

@property
def val(self):
  return self.make_dataset(self.val_df)

@property
def test(self):
  return self.make_dataset(self.test_df)

@property
def example(self):
  """Récupère et met en cache un batch d'exemple `inputs, labels` pour l'affichage."""
  result = getattr(self, '_example', None)
  if result is None:
    # Aucun batch d'exemple trouvé : on en prend un dans le dataset `.train`
    result = next(iter(self.train))
    # Et on le met en cache pour la prochaine fois
    self._example = result
  return result

WindowGenerator.train = train
WindowGenerator.val = val
WindowGenerator.test = test
WindowGenerator.example = example

L'objet `WindowGenerator` vous donne maintenant accès aux objets `tf.data.Dataset`, ce qui vous permet d'itérer facilement sur les données.

La propriété `Dataset.element_spec` vous donne la structure, les types de données et les formes des éléments du dataset.

In [ ]:
# Chaque élément est une paire (inputs, label).
w2.train.element_spec

Itérer sur un `Dataset` produit des batchs concrets :

In [ ]:
for example_inputs, example_labels in w2.train.take(1):
  print(f'Forme des entrées (batch, time, features) : {example_inputs.shape}')
  print(f'Forme des étiquettes (batch, time, features) : {example_labels.shape}')

## Modèles à un pas de temps

Le modèle le plus simple que l'on puisse construire sur ce type de données prédit la valeur d'une seule caractéristique, 1 pas de temps (une heure) dans le futur, à partir des seules conditions actuelles.

Commencez donc par construire des modèles qui prédisent la valeur de `T (degC)` une heure dans le futur.

![Prédire le pas de temps suivant](https://github.com/tensorflow/docs/blob/master/site/en/tutorials/structured_data/images/narrow_window.png?raw=1)

Configurez un objet `WindowGenerator` pour produire ces paires `(input, label)` à un pas de temps :

In [ ]:
single_step_window = WindowGenerator(
    input_width=1, label_width=1, shift=1,
    label_columns=['T (degC)'])
single_step_window

L'objet `window` crée des `tf.data.Dataset` à partir des ensembles d'entraînement, de validation et de test, ce qui permet d'itérer facilement sur des batchs de données.

In [ ]:
for example_inputs, example_labels in single_step_window.train.take(1):
  print(f'Forme des entrées (batch, time, features) : {example_inputs.shape}')
  print(f'Forme des étiquettes (batch, time, features) : {example_labels.shape}')

### Modèle de référence

Avant de construire un modèle entraînable, il est bon d'avoir une performance de référence, point de comparaison avec les modèles plus complexes qui suivront.

Cette première tâche consiste à prédire la température une heure dans le futur, à partir des valeurs actuelles de toutes les caractéristiques. Les valeurs actuelles incluent la température actuelle.

Commencez donc avec un modèle qui renvoie simplement la température actuelle comme prédiction, c'est-à-dire qui prédit « pas de changement ». C'est une référence raisonnable, car la température change lentement. Bien sûr, cette référence fonctionnera moins bien pour une prédiction plus lointaine.

![Renvoyer l'entrée en sortie](https://github.com/tensorflow/docs/blob/master/site/en/tutorials/structured_data/images/baseline.png?raw=1)

In [ ]:
class Baseline(tf.keras.Model):
  def __init__(self, label_index=None):
    super().__init__()
    self.label_index = label_index

  def call(self, inputs):
    if self.label_index is None:
      return inputs
    result = inputs[:, :, self.label_index]
    return result[:, :, tf.newaxis]

Instanciez et évaluez ce modèle :

In [ ]:
baseline = Baseline(label_index=column_indices['T (degC)'])

baseline.compile(loss=tf.keras.losses.MeanSquaredError(),
                 metrics=[tf.keras.metrics.MeanAbsoluteError()])

val_performance = {}
performance = {}
val_performance['Baseline'] = baseline.evaluate(single_step_window.val, return_dict=True)
performance['Baseline'] = baseline.evaluate(single_step_window.test, verbose=0, return_dict=True)

Cela a affiché quelques métriques de performance, mais elles ne donnent pas vraiment idée de la qualité du modèle.

Le `WindowGenerator` a une méthode d'affichage, mais les graphiques ne seront pas très intéressants avec un seul exemple.

Créez donc un `WindowGenerator` plus large, qui génère des fenêtres de 24 heures d'entrées et d'étiquettes consécutives. La nouvelle variable `wide_window` ne change pas le fonctionnement du modèle. Le modèle prédit toujours une heure dans le futur à partir d'un seul pas de temps d'entrée. Ici, l'axe `time` se comporte comme l'axe `batch` : chaque prédiction est faite indépendamment, sans interaction entre les pas de temps :

In [ ]:
wide_window = WindowGenerator(
    input_width=24, label_width=24, shift=1,
    label_columns=['T (degC)'])

wide_window

Cette fenêtre élargie peut être passée directement au même modèle `baseline`, sans modifier le code. C'est possible car les entrées et les étiquettes ont le même nombre de pas de temps, et le modèle de référence renvoie simplement l'entrée en sortie :

![Une prédiction 1 h dans le futur, toutes les heures.](https://github.com/tensorflow/docs/blob/master/site/en/tutorials/structured_data/images/last_window.png?raw=1)

In [ ]:
print("Forme de l'entrée :", wide_window.example[0].shape)
print('Forme de la sortie :', baseline(wide_window.example[0]).shape)

En affichant les prédictions du modèle de référence, on remarque qu'il s'agit simplement des étiquettes décalées d'une heure vers la droite :

In [ ]:
wide_window.plot(baseline)

Dans les graphiques ci-dessus des trois exemples, le modèle à un pas de temps est exécuté sur 24 heures. Quelques explications :

- La ligne bleue `Inputs` montre la température d'entrée à chaque pas de temps. Le modèle reçoit toutes les caractéristiques, ce graphique ne montre que la température.
- Les points verts `Labels` montrent la valeur cible de la prédiction. Ces points sont affichés à l'instant de la prédiction, pas à celui de l'entrée. C'est pourquoi la plage des étiquettes est décalée d'un pas par rapport aux entrées.
- Les croix orange `Predictions` sont les prédictions du modèle pour chaque pas de temps de sortie. Si le modèle prédisait parfaitement, les prédictions tomberaient exactement sur les `Labels`.

### Modèle linéaire

Le modèle **entraînable** le plus simple que l'on puisse appliquer à cette tâche insère une transformation linéaire entre l'entrée et la sortie. Dans ce cas, la sortie d'un pas de temps ne dépend que de ce pas :

![Une prédiction à un pas de temps](https://github.com/tensorflow/docs/blob/master/site/en/tutorials/structured_data/images/narrow_window.png?raw=1)

Une couche `tf.keras.layers.Dense` sans `activation` est un modèle linéaire. La couche ne transforme que le dernier axe des données, de `(batch, time, inputs)` à `(batch, time, units)` ; elle est appliquée indépendamment à chaque élément des axes `batch` et `time`.

In [ ]:
linear = tf.keras.Sequential([
    tf.keras.layers.Dense(units=1)
])

In [ ]:
print("Forme de l'entrée :", single_step_window.example[0].shape)
print('Forme de la sortie :', linear(single_step_window.example[0]).shape)

Ce tutoriel entraîne de nombreux modèles : regroupez donc la procédure d'entraînement dans une fonction :

In [ ]:
MAX_EPOCHS = 20

def compile_and_fit(model, window, patience=2):
  early_stopping = tf.keras.callbacks.EarlyStopping(monitor='val_loss',
                                                    patience=patience,
                                                    mode='min')

  model.compile(loss=tf.keras.losses.MeanSquaredError(),
                optimizer=tf.keras.optimizers.Adam(),
                metrics=[tf.keras.metrics.MeanAbsoluteError()])

  history = model.fit(window.train, epochs=MAX_EPOCHS,
                      validation_data=window.val,
                      callbacks=[early_stopping])
  return history

Entraînez le modèle et évaluez ses performances :

In [ ]:
history = compile_and_fit(linear, single_step_window)

val_performance['Linear'] = linear.evaluate(single_step_window.val, return_dict=True)
performance['Linear'] = linear.evaluate(single_step_window.test, verbose=0, return_dict=True)

Comme le modèle `baseline`, le modèle linéaire peut être appelé sur des batchs de fenêtres larges. Utilisé ainsi, le modèle fait un ensemble de prédictions indépendantes sur des pas de temps consécutifs. L'axe `time` se comporte comme un autre axe `batch`. Il n'y a aucune interaction entre les prédictions des différents pas de temps.

![Une prédiction à un pas de temps](https://github.com/tensorflow/docs/blob/master/site/en/tutorials/structured_data/images/wide_window.png?raw=1)

In [ ]:
print("Forme de l'entrée :", wide_window.example[0].shape)
print('Forme de la sortie :', linear(wide_window.example[0]).shape)

Voici le graphique de ses prédictions d'exemple sur la `wide_window` : notez que dans bien des cas la prédiction est nettement meilleure que la simple température d'entrée, mais que dans quelques cas elle est pire :

In [ ]:
wide_window.plot(linear)

L'un des avantages des modèles linéaires est qu'ils sont relativement simples à interpréter.
Vous pouvez extraire les poids de la couche et visualiser le poids attribué à chaque entrée :

In [ ]:
plt.bar(x = range(len(train_df.columns)),
        height=linear.layers[0].kernel[:,0].numpy())
axis = plt.gca()
axis.set_xticks(range(len(train_df.columns)))
_ = axis.set_xticklabels(train_df.columns, rotation=90)

Parfois, le modèle ne donne même pas le plus grand poids à l'entrée `T (degC)`. C'est l'un des risques de l'initialisation aléatoire.

### Dense

Avant d'appliquer des modèles qui travaillent vraiment sur plusieurs pas de temps, il est utile de vérifier les performances de modèles à un pas d'entrée plus profonds et plus puissants.

Voici un modèle similaire au modèle `linear`, si ce n'est qu'il empile quelques couches `Dense` entre l'entrée et la sortie :

In [ ]:
dense = tf.keras.Sequential([
    tf.keras.layers.Dense(units=64, activation='relu'),
    tf.keras.layers.Dense(units=64, activation='relu'),
    tf.keras.layers.Dense(units=1)
])

history = compile_and_fit(dense, single_step_window)

val_performance['Dense'] = dense.evaluate(single_step_window.val, return_dict=True)
performance['Dense'] = dense.evaluate(single_step_window.test, verbose=0, return_dict=True)

### Dense à plusieurs pas de temps

Un modèle à un seul pas de temps n'a aucun contexte pour les valeurs actuelles de ses entrées. Il ne peut pas voir comment les caractéristiques d'entrée évoluent au cours du temps. Pour résoudre ce problème, le modèle a besoin d'accéder à plusieurs pas de temps pour faire ses prédictions :

![Trois pas de temps sont utilisés pour chaque prédiction.](https://github.com/tensorflow/docs/blob/master/site/en/tutorials/structured_data/images/conv_window.png?raw=1)

Les modèles `baseline`, `linear` et `dense` traitaient chaque pas de temps indépendamment. Ici, le modèle prendra plusieurs pas de temps en entrée pour produire une seule sortie.

Créez un `WindowGenerator` qui produira des batchs d'entrées de trois heures et d'étiquettes d'une heure :

Notez que le paramètre `shift` de `Window` est relatif à la fin des deux fenêtres.

In [ ]:
CONV_WIDTH = 3
conv_window = WindowGenerator(
    input_width=CONV_WIDTH,
    label_width=1,
    shift=1,
    label_columns=['T (degC)'])

conv_window

In [ ]:
conv_window.plot()
plt.suptitle("À partir de 3 heures d'entrées, prédire 1 heure dans le futur.")

Vous pourriez entraîner un modèle `dense` sur une fenêtre à plusieurs pas d'entrée en ajoutant une couche `tf.keras.layers.Flatten` en première couche du modèle :

In [ ]:
multi_step_dense = tf.keras.Sequential([
    # Forme : (time, features) => (time*features)
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(units=32, activation='relu'),
    tf.keras.layers.Dense(units=32, activation='relu'),
    tf.keras.layers.Dense(units=1),
    # Rétablissement de la dimension du temps.
    # Forme : (outputs) => (1, outputs)
    tf.keras.layers.Reshape([1, -1]),
])

In [ ]:
print("Forme de l'entrée :", conv_window.example[0].shape)
print('Forme de la sortie :', multi_step_dense(conv_window.example[0]).shape)

In [ ]:
history = compile_and_fit(multi_step_dense, conv_window)

IPython.display.clear_output()
val_performance['Multi step dense'] = multi_step_dense.evaluate(conv_window.val, return_dict=True)
performance['Multi step dense'] = multi_step_dense.evaluate(conv_window.test, verbose=0, return_dict=True)

In [ ]:
conv_window.plot(multi_step_dense)

Le principal inconvénient de cette approche est que le modèle obtenu ne peut être exécuté que sur des fenêtres d'entrée d'exactement cette forme.

In [ ]:
print("Forme de l'entrée :", wide_window.example[0].shape)
try:
  print('Forme de la sortie :', multi_step_dense(wide_window.example[0]).shape)
except Exception as e:
  print(f'\n{type(e).__name__}:{e}')

Les modèles convolutifs de la section suivante règlent ce problème.

### Réseau de neurones convolutif

Une couche de convolution (`tf.keras.layers.Conv1D`) prend elle aussi plusieurs pas de temps en entrée de chaque prédiction.

Voici le **même** modèle que `multi_step_dense`, réécrit avec une convolution.

Notez les changements :
* La couche `tf.keras.layers.Flatten` et la première `tf.keras.layers.Dense` sont remplacées par une `tf.keras.layers.Conv1D`.
* La couche `tf.keras.layers.Reshape` n'est plus nécessaire, puisque la convolution conserve l'axe du temps dans sa sortie.

In [ ]:
conv_model = tf.keras.Sequential([
    tf.keras.layers.Conv1D(filters=32,
                           kernel_size=(CONV_WIDTH,),
                           activation='relu'),
    tf.keras.layers.Dense(units=32, activation='relu'),
    tf.keras.layers.Dense(units=1),
])

Exécutez-le sur un batch d'exemple pour vérifier que le modèle produit des sorties de la forme attendue :

In [ ]:
print("Modèle convolutif sur `conv_window`")
print("Forme de l'entrée :", conv_window.example[0].shape)
print('Forme de la sortie :', conv_model(conv_window.example[0]).shape)

Entraînez-le et évaluez-le sur la `conv_window` : il devrait donner des performances similaires au modèle `multi_step_dense`.

In [ ]:
history = compile_and_fit(conv_model, conv_window)

IPython.display.clear_output()
val_performance['Conv'] = conv_model.evaluate(conv_window.val, return_dict=True)
performance['Conv'] = conv_model.evaluate(conv_window.test, verbose=0, return_dict=True)

La différence entre ce `conv_model` et le modèle `multi_step_dense` est que le `conv_model` peut être exécuté sur des entrées de n'importe quelle longueur. La couche convolutive est appliquée à une fenêtre glissante d'entrées :

![Exécution d'un modèle convolutif sur une séquence](https://github.com/tensorflow/docs/blob/master/site/en/tutorials/structured_data/images/wide_conv_window.png?raw=1)

Exécuté sur une entrée plus large, il produit une sortie plus large :

In [ ]:
print("Fenêtre large")
print("Forme de l'entrée :", wide_window.example[0].shape)
print('Forme des étiquettes :', wide_window.example[1].shape)
print('Forme de la sortie :', conv_model(wide_window.example[0]).shape)

Notez que la sortie est plus courte que l'entrée. Pour que l'entraînement ou l'affichage fonctionnent, les étiquettes et les prédictions doivent avoir la même longueur. Construisez donc un `WindowGenerator` qui produit des fenêtres larges avec quelques pas de temps d'entrée supplémentaires, pour que les longueurs des étiquettes et des prédictions correspondent :

In [ ]:
LABEL_WIDTH = 24
INPUT_WIDTH = LABEL_WIDTH + (CONV_WIDTH - 1)
wide_conv_window = WindowGenerator(
    input_width=INPUT_WIDTH,
    label_width=LABEL_WIDTH,
    shift=1,
    label_columns=['T (degC)'])

wide_conv_window

In [ ]:
print("Fenêtre convolutive large")
print("Forme de l'entrée :", wide_conv_window.example[0].shape)
print('Forme des étiquettes :', wide_conv_window.example[1].shape)
print('Forme de la sortie :', conv_model(wide_conv_window.example[0]).shape)

Vous pouvez maintenant afficher les prédictions du modèle sur une fenêtre plus large. Notez les 3 pas de temps d'entrée avant la première prédiction. Chaque prédiction s'appuie ici sur les 3 pas de temps précédents :

In [ ]:
wide_conv_window.plot(conv_model)

### Réseau de neurones récurrent

Un réseau de neurones récurrent (RNN) est un type de réseau de neurones bien adapté aux séries temporelles. Les RNN traitent une série temporelle pas à pas, en maintenant un état interne d'un pas de temps au suivant.

Pour en savoir plus, voir le tutoriel [Text generation with an RNN](https://www.tensorflow.org/text/tutorials/text_generation) et le guide [Recurrent Neural Networks (RNN) with Keras](https://www.tensorflow.org/guide/keras/rnn).

Dans ce tutoriel, vous utiliserez une couche RNN appelée *Long Short-Term Memory* (`tf.keras.layers.LSTM`).

Un argument important du constructeur de toutes les couches RNN de Keras, comme `tf.keras.layers.LSTM`, est `return_sequences`. Ce paramètre peut configurer la couche de deux façons :

1. Si `False`, la valeur par défaut, la couche ne renvoie que la sortie du dernier pas de temps, ce qui laisse au modèle le temps de « chauffer » son état interne avant de faire une seule prédiction :

![Un LSTM qui chauffe puis fait une seule prédiction](https://github.com/tensorflow/docs/blob/master/site/en/tutorials/structured_data/images/lstm_1_window.png?raw=1)

2. Si `True`, la couche renvoie une sortie pour chaque entrée. C'est utile pour :
  * Empiler des couches RNN.
  * Entraîner un modèle sur plusieurs pas de temps simultanément.

![Un LSTM qui fait une prédiction après chaque pas de temps](https://github.com/tensorflow/docs/blob/master/site/en/tutorials/structured_data/images/lstm_many_window.png?raw=1)

In [ ]:
lstm_model = tf.keras.models.Sequential([
    # Forme [batch, time, features] => [batch, time, lstm_units]
    tf.keras.layers.LSTM(32, return_sequences=True),
    # Forme => [batch, time, features]
    tf.keras.layers.Dense(units=1)
])

Avec `return_sequences=True`, le modèle peut être entraîné sur 24 heures de données à la fois.

Note : cela donne une vision pessimiste des performances du modèle. Au premier pas de temps, le modèle n'a pas accès aux pas précédents et ne peut donc pas faire mieux que les modèles simples `linear` et `dense` vus plus haut.

In [ ]:
print("Forme de l'entrée :", wide_window.example[0].shape)
print('Forme de la sortie :', lstm_model(wide_window.example[0]).shape)

In [ ]:
history = compile_and_fit(lstm_model, wide_window)

IPython.display.clear_output()
val_performance['LSTM'] = lstm_model.evaluate(wide_window.val, return_dict=True)
performance['LSTM'] = lstm_model.evaluate(wide_window.test, verbose=0, return_dict=True)

In [ ]:
wide_window.plot(lstm_model)

### Performances

Sur ce jeu de données, chaque modèle fait en général un peu mieux que le précédent :

In [ ]:
cm = lstm_model.metrics[1]
cm.metrics

In [ ]:
val_performance

In [ ]:
x = np.arange(len(performance))
width = 0.3
metric_name = 'mean_absolute_error'
val_mae = [v[metric_name] for v in val_performance.values()]
test_mae = [v[metric_name] for v in performance.values()]

plt.ylabel('mean_absolute_error [T (degC), normalisée]')
plt.bar(x - 0.17, val_mae, width, label='Validation')
plt.bar(x + 0.17, test_mae, width, label='Test')
plt.xticks(ticks=x, labels=performance.keys(),
           rotation=45)
_ = plt.legend()

In [ ]:
for name, value in performance.items():
  print(f'{name:12s}: {value[metric_name]:0.4f}')

### Modèles à plusieurs sorties

Jusqu'ici, tous les modèles prédisaient une seule caractéristique de sortie, `T (degC)`, pour un seul pas de temps.

Tous ces modèles peuvent être convertis pour prédire plusieurs caractéristiques, simplement en changeant le nombre d'unités de la couche de sortie et en ajustant les fenêtres d'entraînement pour inclure toutes les caractéristiques dans les `labels` (`example_labels`) :

In [ ]:
single_step_window = WindowGenerator(
    # `WindowGenerator` renvoie toutes les caractéristiques comme étiquettes si
    # l'argument `label_columns` n'est pas précisé.
    input_width=1, label_width=1, shift=1)

wide_window = WindowGenerator(
    input_width=24, label_width=24, shift=1)

for example_inputs, example_labels in wide_window.train.take(1):
  print(f'Forme des entrées (batch, time, features) : {example_inputs.shape}')
  print(f'Forme des étiquettes (batch, time, features) : {example_labels.shape}')

Notez ci-dessus que l'axe `features` des étiquettes a maintenant la même profondeur que les entrées, au lieu de `1`.

#### Modèle de référence

Le même modèle de référence (`Baseline`) peut être utilisé ici, mais cette fois en répétant toutes les caractéristiques au lieu d'en sélectionner une avec `label_index` :

In [ ]:
baseline = Baseline()
baseline.compile(loss=tf.keras.losses.MeanSquaredError(),
                 metrics=[tf.keras.metrics.MeanAbsoluteError()])

In [ ]:
val_performance = {}
performance = {}
val_performance['Baseline'] = baseline.evaluate(wide_window.val, return_dict=True)
performance['Baseline'] = baseline.evaluate(wide_window.test, verbose=0, return_dict=True)

#### Dense

In [ ]:
dense = tf.keras.Sequential([
    tf.keras.layers.Dense(units=64, activation='relu'),
    tf.keras.layers.Dense(units=64, activation='relu'),
    tf.keras.layers.Dense(units=num_features)
])

In [ ]:
history = compile_and_fit(dense, single_step_window)

IPython.display.clear_output()
val_performance['Dense'] = dense.evaluate(single_step_window.val, return_dict=True)
performance['Dense'] = dense.evaluate(single_step_window.test, verbose=0, return_dict=True)

#### RNN

In [ ]:
%%time
wide_window = WindowGenerator(
    input_width=24, label_width=24, shift=1)

lstm_model = tf.keras.models.Sequential([
    # Forme [batch, time, features] => [batch, time, lstm_units]
    tf.keras.layers.LSTM(32, return_sequences=True),
    # Forme => [batch, time, features]
    tf.keras.layers.Dense(units=num_features)
])

history = compile_and_fit(lstm_model, wide_window)

IPython.display.clear_output()
val_performance['LSTM'] = lstm_model.evaluate( wide_window.val, return_dict=True)
performance['LSTM'] = lstm_model.evaluate( wide_window.test, verbose=0, return_dict=True)

print()

<a id="residual"></a>

#### Pour aller plus loin : connexions résiduelles

Le modèle `Baseline` vu plus haut tirait parti du fait que la séquence ne change pas radicalement d'un pas de temps au suivant. Jusqu'ici, tous les modèles de ce tutoriel ont été initialisés aléatoirement, puis ont dû apprendre que la sortie est un petit changement par rapport au pas de temps précédent.

On peut contourner ce problème avec une initialisation soignée, mais il est plus simple de l'intégrer à la structure du modèle.

En analyse de séries temporelles, il est courant de construire des modèles qui, au lieu de prédire la valeur suivante, prédisent comment la valeur va changer au pas de temps suivant. De même, les <a href="https://arxiv.org/abs/1512.03385" class="external">réseaux résiduels</a> (ResNets) désignent en deep learning des architectures où chaque couche s'ajoute au résultat accumulé du modèle.

C'est ainsi qu'on tire parti de la connaissance que le changement doit être petit.

![Un modèle avec une connexion résiduelle](https://github.com/tensorflow/docs/blob/master/site/en/tutorials/structured_data/images/residual.png?raw=1)

En pratique, cela initialise le modèle pour qu'il corresponde au `Baseline`. Pour cette tâche, cela aide les modèles à converger plus vite, avec des performances légèrement meilleures.

Cette approche peut être combinée avec n'importe quel modèle de ce tutoriel.

Ici, elle est appliquée au modèle LSTM : notez l'utilisation de `tf.initializers.zeros` pour que les changements prédits initialement soient petits et n'écrasent pas la connexion résiduelle. Il n'y a pas ici de problème de brisure de symétrie pour les gradients, puisque les `zeros` ne sont utilisés que sur la dernière couche.

In [ ]:
class ResidualWrapper(tf.keras.Model):
  def __init__(self, model):
    super().__init__()
    self.model = model

  def call(self, inputs, *args, **kwargs):
    delta = self.model(inputs, *args, **kwargs)

    # La prédiction pour chaque pas de temps est l'entrée
    # du pas de temps précédent plus l'écart
    # calculé par le modèle.
    return inputs + delta

In [ ]:
%%time
residual_lstm = ResidualWrapper(
    tf.keras.Sequential([
    tf.keras.layers.LSTM(32, return_sequences=True),
    tf.keras.layers.Dense(
        num_features,
        # Les écarts prédits doivent être petits au départ.
        # On initialise donc la couche de sortie avec des zéros.
        kernel_initializer=tf.initializers.zeros())
]))

history = compile_and_fit(residual_lstm, wide_window)

IPython.display.clear_output()
val_performance['Residual LSTM'] = residual_lstm.evaluate(wide_window.val, return_dict=True)
performance['Residual LSTM'] = residual_lstm.evaluate(wide_window.test, verbose=0, return_dict=True)
print()

#### Performances

Voici les performances globales de ces modèles à plusieurs sorties.

In [ ]:
x = np.arange(len(performance))
width = 0.3

metric_name = 'mean_absolute_error'
val_mae = [v[metric_name] for v in val_performance.values()]
test_mae = [v[metric_name] for v in performance.values()]

plt.bar(x - 0.17, val_mae, width, label='Validation')
plt.bar(x + 0.17, test_mae, width, label='Test')
plt.xticks(ticks=x, labels=performance.keys(),
           rotation=45)
plt.ylabel('MAE (moyenne sur toutes les sorties)')
_ = plt.legend()

In [ ]:
for name, value in performance.items():
  print(f'{name:15s}: {value[metric_name]:0.4f}')

Les performances ci-dessus sont moyennées sur toutes les sorties du modèle.

## Modèles à plusieurs pas de temps

Les modèles à une et à plusieurs sorties des sections précédentes faisaient des **prédictions d'un seul pas de temps**, une heure dans le futur.

Cette section montre comment étendre ces modèles pour faire des **prédictions de plusieurs pas de temps**.

Dans une prédiction à plusieurs pas, le modèle doit apprendre à prédire une plage de valeurs futures. Ainsi, contrairement à un modèle à un pas, qui ne prédit qu'un seul point futur, un modèle à plusieurs pas prédit une séquence de valeurs futures.

Il y a grosso modo deux approches :

1. Les prédictions en une fois (*single-shot*), où toute la série temporelle est prédite d'un coup.
2. Les prédictions autorégressives, où le modèle ne fait que des prédictions à un pas, et où sa sortie est renvoyée en entrée.

Dans cette section, tous les modèles prédiront **toutes les caractéristiques pour tous les pas de temps de sortie**.

Pour le modèle à plusieurs pas, les données d'entraînement sont toujours des exemples horaires. En revanche, ici, les modèles apprendront à prédire 24 heures dans le futur à partir de 24 heures de passé.

Voici un objet `Window` qui génère ces tranches à partir du jeu de données :

In [ ]:
OUT_STEPS = 24
multi_window = WindowGenerator(input_width=24,
                               label_width=OUT_STEPS,
                               shift=OUT_STEPS)

multi_window.plot()
multi_window

### Modèles de référence

Une référence simple pour cette tâche consiste à répéter le dernier pas de temps d'entrée autant de fois que de pas de temps de sortie demandés :

![Répéter la dernière entrée pour chaque pas de sortie](https://github.com/tensorflow/docs/blob/master/site/en/tutorials/structured_data/images/multistep_last.png?raw=1)

In [ ]:
class MultiStepLastBaseline(tf.keras.Model):
  def call(self, inputs):
    return tf.tile(inputs[:, -1:, :], [1, OUT_STEPS, 1])

last_baseline = MultiStepLastBaseline()
last_baseline.compile(loss=tf.keras.losses.MeanSquaredError(),
                      metrics=[tf.keras.metrics.MeanAbsoluteError()])

multi_val_performance = {}
multi_performance = {}

multi_val_performance['Last'] = last_baseline.evaluate(multi_window.val, return_dict=True)
multi_performance['Last'] = last_baseline.evaluate(multi_window.test, verbose=0, return_dict=True)
multi_window.plot(last_baseline)

Comme cette tâche consiste à prédire 24 heures dans le futur à partir de 24 heures de passé, une autre approche simple consiste à répéter la veille, en supposant que demain sera similaire :

![Répéter la veille](https://github.com/tensorflow/docs/blob/master/site/en/tutorials/structured_data/images/multistep_repeat.png?raw=1)

In [ ]:
class RepeatBaseline(tf.keras.Model):
  def call(self, inputs):
    return inputs

repeat_baseline = RepeatBaseline()
repeat_baseline.compile(loss=tf.keras.losses.MeanSquaredError(),
                        metrics=[tf.keras.metrics.MeanAbsoluteError()])

multi_val_performance['Repeat'] = repeat_baseline.evaluate(multi_window.val, return_dict=True)
multi_performance['Repeat'] = repeat_baseline.evaluate(multi_window.test, verbose=0, return_dict=True)
multi_window.plot(repeat_baseline)

### Modèles en une fois (*single-shot*)

Une approche de haut niveau de ce problème consiste à utiliser un modèle « en une fois », qui prédit toute la séquence en une seule étape.

Cela peut s'implémenter efficacement avec une couche `tf.keras.layers.Dense` à `OUT_STEPS*features` unités de sortie. Le modèle n'a plus qu'à redimensionner cette sortie en `(OUTPUT_STEPS, features)`.

#### Linéaire

Un modèle linéaire simple fondé sur le dernier pas de temps d'entrée fait mieux que les deux références, mais manque de puissance. Le modèle doit prédire `OUTPUT_STEPS` pas de temps à partir d'un seul pas de temps d'entrée, par une projection linéaire. Il ne peut capturer qu'une tranche de faible dimension du comportement, sans doute fondée surtout sur l'heure de la journée et le moment de l'année.

![Prédire tous les pas de temps à partir du dernier](https://github.com/tensorflow/docs/blob/master/site/en/tutorials/structured_data/images/multistep_dense.png?raw=1)

In [ ]:
multi_linear_model = tf.keras.Sequential([
    # On prend le dernier pas de temps.
    # Forme [batch, time, features] => [batch, 1, features]
    tf.keras.layers.Lambda(lambda x: x[:, -1:, :]),
    # Forme => [batch, 1, out_steps*features]
    tf.keras.layers.Dense(OUT_STEPS*num_features,
                          kernel_initializer=tf.initializers.zeros()),
    # Forme => [batch, out_steps, features]
    tf.keras.layers.Reshape([OUT_STEPS, num_features])
])

history = compile_and_fit(multi_linear_model, multi_window)

IPython.display.clear_output()
multi_val_performance['Linear'] = multi_linear_model.evaluate(multi_window.val, return_dict=True)
multi_performance['Linear'] = multi_linear_model.evaluate(multi_window.test, verbose=0, return_dict=True)
multi_window.plot(multi_linear_model)

#### Dense

Ajouter une couche `tf.keras.layers.Dense` entre l'entrée et la sortie donne plus de puissance au modèle linéaire, mais il ne s'appuie toujours que sur un seul pas de temps d'entrée.

In [ ]:
multi_dense_model = tf.keras.Sequential([
    # On prend le dernier pas de temps.
    # Forme [batch, time, features] => [batch, 1, features]
    tf.keras.layers.Lambda(lambda x: x[:, -1:, :]),
    # Forme => [batch, 1, dense_units]
    tf.keras.layers.Dense(512, activation='relu'),
    # Forme => [batch, out_steps*features]
    tf.keras.layers.Dense(OUT_STEPS*num_features,
                          kernel_initializer=tf.initializers.zeros()),
    # Forme => [batch, out_steps, features]
    tf.keras.layers.Reshape([OUT_STEPS, num_features])
])

history = compile_and_fit(multi_dense_model, multi_window)

IPython.display.clear_output()
multi_val_performance['Dense'] = multi_dense_model.evaluate(multi_window.val, return_dict=True)
multi_performance['Dense'] = multi_dense_model.evaluate(multi_window.test, verbose=0, return_dict=True)
multi_window.plot(multi_dense_model)

#### CNN

Un modèle convolutif fait ses prédictions à partir d'un historique de largeur fixe, ce qui peut donner de meilleures performances que le modèle dense, puisqu'il peut voir comment les choses évoluent au cours du temps :

![Un modèle convolutif voit comment les choses évoluent au cours du temps](https://github.com/tensorflow/docs/blob/master/site/en/tutorials/structured_data/images/multistep_conv.png?raw=1)

In [ ]:
CONV_WIDTH = 3
multi_conv_model = tf.keras.Sequential([
    # Forme [batch, time, features] => [batch, CONV_WIDTH, features]
    tf.keras.layers.Lambda(lambda x: x[:, -CONV_WIDTH:, :]),
    # Forme => [batch, 1, conv_units]
    tf.keras.layers.Conv1D(256, activation='relu', kernel_size=(CONV_WIDTH)),
    # Forme => [batch, 1,  out_steps*features]
    tf.keras.layers.Dense(OUT_STEPS*num_features,
                          kernel_initializer=tf.initializers.zeros()),
    # Forme => [batch, out_steps, features]
    tf.keras.layers.Reshape([OUT_STEPS, num_features])
])

history = compile_and_fit(multi_conv_model, multi_window)

IPython.display.clear_output()

multi_val_performance['Conv'] = multi_conv_model.evaluate(multi_window.val, return_dict=True)
multi_performance['Conv'] = multi_conv_model.evaluate(multi_window.test, verbose=0, return_dict=True)
multi_window.plot(multi_conv_model)

#### RNN

Un modèle récurrent peut apprendre à utiliser un long historique d'entrées, s'il est pertinent pour les prédictions du modèle. Ici, le modèle accumule un état interne pendant 24 heures, avant de faire une seule prédiction pour les 24 heures suivantes.

Dans ce format en une fois, le LSTM n'a besoin de produire une sortie qu'au dernier pas de temps : réglez donc `return_sequences=False` dans `tf.keras.layers.LSTM`.

![Le LSTM accumule un état sur la fenêtre d'entrée et fait une seule prédiction pour les 24 heures suivantes](https://github.com/tensorflow/docs/blob/master/site/en/tutorials/structured_data/images/multistep_lstm.png?raw=1)

In [ ]:
multi_lstm_model = tf.keras.Sequential([
    # Forme [batch, time, features] => [batch, lstm_units].
    # Ajouter des `lstm_units` ne fait que sur-apprendre plus vite.
    tf.keras.layers.LSTM(32, return_sequences=False),
    # Forme => [batch, out_steps*features].
    tf.keras.layers.Dense(OUT_STEPS*num_features,
                          kernel_initializer=tf.initializers.zeros()),
    # Forme => [batch, out_steps, features].
    tf.keras.layers.Reshape([OUT_STEPS, num_features])
])

history = compile_and_fit(multi_lstm_model, multi_window)

IPython.display.clear_output()

multi_val_performance['LSTM'] = multi_lstm_model.evaluate(multi_window.val, return_dict=True)
multi_performance['LSTM'] = multi_lstm_model.evaluate(multi_window.test, verbose=0, return_dict=True)
multi_window.plot(multi_lstm_model)

### Pour aller plus loin : modèle autorégressif

Les modèles ci-dessus prédisent tous la séquence de sortie complète en une seule étape.

Dans certains cas, il peut être utile que le modèle décompose cette prédiction en pas de temps individuels. La sortie du modèle peut alors lui être renvoyée à chaque pas, et chaque prédiction peut être conditionnée par la précédente, comme dans le classique <a href="https://arxiv.org/abs/1308.0850" class="external">Generating Sequences With Recurrent Neural Networks</a>.

Un avantage évident de ce type de modèle est qu'il peut être configuré pour produire des sorties de longueur variable.

Vous pourriez prendre n'importe lequel des modèles à un pas et plusieurs sorties entraînés dans la première moitié de ce tutoriel et l'exécuter dans une boucle de rétroaction autorégressive, mais ici, vous vous concentrerez sur la construction d'un modèle explicitement entraîné pour cela.

![Renvoyer la sortie d'un modèle en entrée](https://github.com/tensorflow/docs/blob/master/site/en/tutorials/structured_data/images/multistep_autoregressive.png?raw=1)

#### RNN

Ce tutoriel ne construit qu'un modèle RNN autorégressif, mais ce schéma peut s'appliquer à n'importe quel modèle conçu pour produire un seul pas de temps en sortie.

Le modèle aura la même forme de base que les modèles LSTM à un pas vus plus haut : une couche `tf.keras.layers.LSTM` suivie d'une couche `tf.keras.layers.Dense` qui convertit les sorties de la couche `LSTM` en prédictions du modèle.

Une `tf.keras.layers.LSTM` est une `tf.keras.layers.LSTMCell` enveloppée dans la couche de plus haut niveau `tf.keras.layers.RNN`, qui gère pour vous l'état et les résultats de séquence (voir le guide [Recurrent Neural Networks (RNN) with Keras](https://www.tensorflow.org/guide/keras/rnn) pour les détails).

Ici, le modèle doit gérer manuellement les entrées de chaque pas : il utilise donc directement `tf.keras.layers.LSTMCell`, l'interface de plus bas niveau, à un pas de temps.

In [ ]:
class FeedBack(tf.keras.Model):
  def __init__(self, units, out_steps):
    super().__init__()
    self.out_steps = out_steps
    self.units = units
    self.lstm_cell = tf.keras.layers.LSTMCell(units)
    # On enveloppe aussi la LSTMCell dans un RNN pour simplifier la méthode `warmup`.
    self.lstm_rnn = tf.keras.layers.RNN(self.lstm_cell, return_state=True)
    self.dense = tf.keras.layers.Dense(num_features)

In [ ]:
feedback_model = FeedBack(units=32, out_steps=OUT_STEPS)

La première méthode dont ce modèle a besoin est une méthode `warmup`, qui initialise son état interne à partir des entrées. Une fois le modèle entraîné, cet état capture les parties pertinentes de l'historique des entrées. C'est l'équivalent du modèle `LSTM` à un pas vu plus haut :

In [ ]:
def warmup(self, inputs):
  # inputs.shape => (batch, time, features)
  # x.shape => (batch, lstm_units)
  x, *state = self.lstm_rnn(inputs)

  # predictions.shape => (batch, features)
  prediction = self.dense(x)
  return prediction, state

FeedBack.warmup = warmup

Cette méthode renvoie une prédiction à un pas de temps et l'état interne du `LSTM` :

In [ ]:
prediction, state = feedback_model.warmup(multi_window.example[0])
prediction.shape

Avec l'état du `RNN` et une prédiction initiale, vous pouvez maintenant continuer à itérer le modèle, en lui renvoyant à chaque pas ses prédictions en entrée.

L'approche la plus simple pour collecter les prédictions de sortie consiste à utiliser une liste Python et un `tf.stack` après la boucle.

Note : empiler une liste Python ainsi ne fonctionne qu'en exécution *eager*, avec `Model.compile(..., run_eagerly=True)` pour l'entraînement, ou avec une sortie de longueur fixe. Pour une longueur de sortie dynamique, il faudrait utiliser un `tf.TensorArray` au lieu d'une liste Python, et `tf.range` au lieu du `range` de Python.

In [ ]:
def call(self, inputs, training=None):
  # Utiliser un TensorArray pour capturer les sorties déroulées dynamiquement.
  predictions = []
  # Initialisation de l'état du LSTM.
  prediction, state = self.warmup(inputs)

  # Ajout de la première prédiction.
  predictions.append(prediction)

  # Exécution des autres pas de prédiction.
  for n in range(1, self.out_steps):
    # Utilisation de la dernière prédiction comme entrée.
    x = prediction
    # Exécution d'un pas du lstm.
    x, state = self.lstm_cell(x, states=state,
                              training=training)
    # Conversion de la sortie du lstm en prédiction.
    prediction = self.dense(x)
    # Ajout de la prédiction à la sortie.
    predictions.append(prediction)

  # predictions.shape => (time, batch, features)
  predictions = tf.stack(predictions)
  # predictions.shape => (batch, time, features)
  predictions = tf.transpose(predictions, [1, 0, 2])
  return predictions

FeedBack.call = call

Testez ce modèle sur les entrées d'exemple :

In [ ]:
print('Forme de la sortie (batch, time, features) : ', feedback_model(multi_window.example[0]).shape)

Entraînez maintenant le modèle :

In [ ]:
history = compile_and_fit(feedback_model, multi_window)

IPython.display.clear_output()

multi_val_performance['AR LSTM'] = feedback_model.evaluate(multi_window.val, return_dict=True)
multi_performance['AR LSTM'] = feedback_model.evaluate(multi_window.test, verbose=0, return_dict=True)
multi_window.plot(feedback_model)

### Performances

Sur ce problème, les gains diminuent clairement avec la complexité du modèle :

In [ ]:
x = np.arange(len(multi_performance))
width = 0.3

metric_name = 'mean_absolute_error'
val_mae = [v[metric_name] for v in multi_val_performance.values()]
test_mae = [v[metric_name] for v in multi_performance.values()]

plt.bar(x - 0.17, val_mae, width, label='Validation')
plt.bar(x + 0.17, test_mae, width, label='Test')
plt.xticks(ticks=x, labels=multi_performance.keys(),
           rotation=45)
plt.ylabel(f'MAE (moyenne sur tous les instants et toutes les sorties)')
_ = plt.legend()

Les métriques des modèles à plusieurs sorties de la première moitié de ce tutoriel montrent les performances moyennées sur toutes les caractéristiques de sortie. Ces performances-ci sont similaires, mais également moyennées sur les pas de temps de sortie.

In [ ]:
for name, value in multi_performance.items():
  print(f'{name:8s}: {value[metric_name]:0.4f}')

Les gains obtenus en passant d'un modèle dense à des modèles convolutifs et récurrents ne sont que de quelques pourcents (quand il y en a), et le modèle autorégressif est clairement moins bon. Ces approches plus complexes ne valent donc peut-être pas la peine sur **ce** problème, mais il n'y avait aucun moyen de le savoir sans essayer, et ces modèles pourraient être utiles pour **votre** problème.

## Pour aller plus loin

Ce tutoriel était une introduction rapide à la prévision de séries temporelles avec TensorFlow.

Pour en savoir plus :

- Le chapitre 15 de <a href="https://www.oreilly.com/library/view/hands-on-machine-learning/9781492032632/" class="external">Hands-on Machine Learning with Scikit-Learn, Keras, and TensorFlow</a>, 2e édition.
- Le chapitre 6 de <a href="https://www.manning.com/books/deep-learning-with-python" class="external">Deep Learning with Python</a>.
- La leçon 8 de l'<a href="https://www.udacity.com/course/intro-to-tensorflow-for-deep-learning--ud187" class="external">introduction à TensorFlow pour le deep learning d'Udacity</a>, avec ses <a href="https://github.com/tensorflow/examples/tree/master/courses/udacity_intro_to_tensorflow_for_deep_learning" class="external">notebooks d'exercices</a>.

Rappelez-vous aussi que vous pouvez implémenter n'importe quel <a href="https://otexts.com/fpp2/index.html" class="external">modèle classique de séries temporelles</a> dans TensorFlow : ce tutoriel se concentre simplement sur les fonctionnalités intégrées de TensorFlow.